In [ ]:
!python -m pip install -U "imagecodecs[all]"
!pip install openslide-bin
!pip install openslide-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 99.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
import cv2
import os
from skimage import io as skio
import matplotlib.patches as patches
import os

# There are two ways to load the data from the PANDA dataset:
# Option 1: Load images using openslide
# Option 2: Load images using skimage (requires that tifffile is installed)
import skimage.io
import openslide
# General packages
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import PIL
from IPython.display import Image, display

# Plotly for the interactive viewer (see last section)
import plotly.graph_objs as go

In [ ]:
def get_tiles(image, tile_size, n_tiles, padding=0):
    """
        Esta função divide a imagem em pequenos blocos ("tiles") de tamanho
        tile_size × tile_size e garante que tenha pelo menos n_tiles blocos.

        Parâmetros:
            - image: imagem a ser dividida
            - tile_size: tamanho do bloco
            - n_tiles: número mínimo de blocos
            - padding: Define se utiliza ou não padding (1 ou 0)

        Retorno:
            - image_reshaped: lista de blocos (tiles)
            - n_tiles_with_info: booleano que indica se há informações suficientes em todos os blocos
    """
    height, width, channels = image.shape # Pegando as dimensões da imagem
    # Calcula quando de padding é necessário para que a imagem seja dividida em blocos de tamanho tile_size
    pad_height = (tile_size - height % tile_size) % tile_size + ((tile_size * padding) // 2)
    pad_width = (tile_size - width % tile_size) % tile_size + ((tile_size * padding) // 2)

    # Adiciona o padding na imagem com o valor 255 (branco)
    image_with_padding = np.pad(image, [[pad_height // 2, pad_height - pad_height // 2], [pad_width // 2, pad_width - pad_width // 2], [0, 0]], constant_values=255,)

    # Calcula o número de blocos ao longo do eixo x e y
    n_blocks_height = image_with_padding.shape[0] // tile_size # Númmero de blocos do eixo y
    n_blocks_width = image_with_padding.shape[1] // tile_size # Número de blocos do eixo x

    # Remodela a imagem para ficar no formato (n_blocos_altura, tile_size, n_blocos_largura, tile_size, 3 canais)
    image_reshaped = image_with_padding.reshape(n_blocks_height, tile_size, n_blocks_width, tile_size, channels)

    # Transpose deixa a imagem no formato (n_blocos_altura, n_blocos_largura, tile_size, tile_size, 3 canais)
    image_reshaped = image_reshaped.transpose(0, 2, 1, 3, 4)

    # reorganiza a imagem para agrupar os blocos de forma que cada
    # bloco seja tratado como uma imagem individual de tamanho tile_size × tile_size.
    # A imagem final fica (n_blocos_altura * n_blocos_largura, tile_size, tile_size, 3 canais)
    image_reshaped = image_reshaped.reshape(-1, tile_size, tile_size, channels)

    max_value_p_block = tile_size ** 2 * channels * 255 # Valor máximo da soma de pixels de >>um<< bloco

    #Calcula o número de blocos que contêm "informação" (ou seja, que não são totalmente brancos).
    #A comparação é feita verificando se a soma dos valores de pixels em cada bloco é menor do que
    #o valor máximo (que seria um bloco completamente branco).
    n_tiles_with_info = (image_reshaped.reshape(image_reshaped.shape[0], -1).sum(1) < max_value_p_block).sum()

    # Se o número de blocos gerados for menor que n_tiles, adiciona blocos brancos para garantir que haja
    # pelo menos n_tiles blocos.
    if len(image_reshaped) < n_tiles:
        image_reshaped = np.pad(image_reshaped, [[0, n_tiles - len(image_reshaped)], [0, 0], [0, 0], [0, 0]], constant_values=255,)

    # NESSA PARTE OS BLOCOS SÃO ORDENADOS COM BASE NA QUANTIDADE DE INFORMAÇÃO QUE CONTÊM
    indexes = np.argsort(image_reshaped.reshape(image_reshaped.shape[0], -1).sum(-1))[:n_tiles]
    image_reshaped = image_reshaped[indexes]

    # Retorna a lista dos blocos (image_reshaped) e um booleano que indica se há informações suficientes em todos os blocos
    # (se o número de blocos com informação é maior ou igual a n_tiles).
    return image_reshaped, n_tiles_with_info >= n_tiles


import numpy as np

def concat_tiles(tiles):
    """
        Esta função concatena os blocos (tiles) de uma imagem, organizando-os
        pela quantidade de informação (blocos menos brancos primeiro).

        Parâmetros:
            - tiles: lista de blocos.

        Retorno:
            - image: imagem de blocos reconstruída.
    """
    n_tiles = tiles.shape[0]
    height, width, channels = tiles.shape[1:]

    n_tiles_per_row = int(np.sqrt(n_tiles))
    n_tiles_per_col = n_tiles_per_row

    if n_tiles_per_row ** 2 != n_tiles:
        print("Não é possível reconstruir a imagem, número de blocos não é um quadrado perfeito")
        return None

    # Ordenar os blocos pelo menor valor da soma dos pixels (menos branco primeiro)
    tile_sums = tiles.reshape(n_tiles, -1).sum(axis=1)
    sorted_indexes = np.argsort(tile_sums)  # Ordena dos menos brancos para os mais brancos
    tiles_sorted = tiles[sorted_indexes]

    # Reorganiza os blocos para que formem uma matriz quadrada antes da concatenação
    tiles_sorted = tiles_sorted.reshape(n_tiles_per_row, n_tiles_per_col, height, width, channels)
    tiles_sorted = tiles_sorted.transpose(0, 2, 1, 3, 4).reshape(n_tiles_per_row * height, n_tiles_per_col * width, channels)

    return tiles_sorted

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import cv2
import numpy as np
import openslide
from threading import Lock

path_source = "/content/drive/MyDrive/Doutorado/kaggle/dataset/train_images"
path_target = "/content/drive/MyDrive/Doutorado/kaggle/dataset/tiles"

# Lock para controlar o acesso ao slide (OpenSlide não é thread-safe)
slide_lock = Lock()

def process_image(image):
    tiles, OK = get_tiles(image, 256, 36, 0)
    full_image = concat_tiles(tiles)
    return cv2.cvtColor(full_image, cv2.COLOR_BGR2RGB)

def process_single_image(args):
    index, filename, total_files, files_to_process = args
    name = f"{filename.split('.')[0]}.png"
    output_path = f"{path_target}/{name}"

    # Verificar se já existe
    if os.path.exists(output_path):
        return f"⊙ {name} (já existe, pulado)"

    print(f"{index+1}/{total_files} ({files_to_process} restantes): {name}")

    try:
        # Leitura do slide (com lock para segurança)
        with slide_lock:
            slide = openslide.OpenSlide(f"{path_source}/{filename}")
            full_size = slide.level_dimensions[1]
            image = slide.read_region((0, 0), 1, full_size)
            slide.close()

        # Conversão para numpy array
        image = np.array(image)
        image = image[:, :, :3]

        # Processamento
        save_image = process_image(image)

        # Salvamento
        cv2.imwrite(output_path, save_image)

        return f"✓ {name}"
    except Exception as e:
        return f"✗ {name}: {str(e)}"

# Criar pasta de destino se não existir
os.makedirs(path_target, exist_ok=True)

# Preparar lista de arquivos
print("Analisando arquivos...")
all_files = [f for f in os.listdir(path_source) if f.endswith('.tiff')]
total_files = len(all_files)

# Verificar quais arquivos já foram processados
existing_files = set(os.listdir(path_target))
files_to_process = []

for filename in all_files:
    output_name = f"{filename.split('.')[0]}.png"
    if output_name not in existing_files:
        files_to_process.append(filename)

files_already_done = total_files - len(files_to_process)

print(f"\n{'='*60}")
print(f"Total de arquivos fonte: {total_files}")
print(f"Já processados: {files_already_done}")
print(f"Restantes para processar: {len(files_to_process)}")
print(f"{'='*60}\n")

if len(files_to_process) == 0:
    print("✓ Todos os arquivos já foram processados!")
else:
    # Preparar tarefas apenas para arquivos não processados
    tasks = [
        (all_files.index(filename), filename, total_files, len(files_to_process))
        for filename in files_to_process
    ]

    # Processar em paralelo
    print("Iniciando processamento...\n")
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_single_image, task): task for task in tasks}

        for future in as_completed(futures):
            result = future.result()
            if not result.startswith("⊙"):  # Não imprimir os pulados
                print(result)

    print(f"\n{'='*60}")
    print(f"Processamento concluído!")
    print(f"Novos arquivos gerados: {len(files_to_process)}")
    print(f"Total no destino: {total_files}")
    print(f"{'='*60}")

Analisando arquivos...

Total de arquivos fonte: 10616
Já processados: 10610
Restantes para processar: 6

Iniciando processamento...

484/10616 (6 restantes): f43e3bd17a33c0ce73efbf07f711c328.png
10140/10616 (6 restantes): 0593354ca8cb549c63f547c29f3a7ff1.png
10196/10616 (6 restantes): 01e5bb47fb34f075f87cbfbc8d630124.png
7684/10616 (6 restantes): 52754aa5f7eb637258abda6dac50201b.png
10212/10616 (6 restantes): 005e66f06bce9c2e49142536caf2f6ee.png
10413/10616 (6 restantes): 0a93d9a674e358ba32e4e2ec320aefc5.png
✗ f43e3bd17a33c0ce73efbf07f711c328.png: Unsupported or missing image file
✗ 0593354ca8cb549c63f547c29f3a7ff1.png: Unsupported or missing image file
✗ 01e5bb47fb34f075f87cbfbc8d630124.png: Unsupported or missing image file
✗ 52754aa5f7eb637258abda6dac50201b.png: Unsupported or missing image file
✗ 005e66f06bce9c2e49142536caf2f6ee.png: Unsupported or missing image file
✗ 0a93d9a674e358ba32e4e2ec320aefc5.png: Unsupported or missing image file

Processamento concluído!
Novos arquivos 

In [ ]:
# Move the zip file to Google Drive
!mv /content/tiles.zip /content/drive/MyDrive/Doutorado/kaggle/tiles.zip

print("The file 'tiles.zip' has been moved to your Google Drive.")

mv: error writing '/content/drive/MyDrive/Doutorado/kaggle/tiles.zip': No space left on device
The file 'tiles.zip' has been moved to your Google Drive.


In [ ]:
import openslide
from PIL import Image
from IPython.display import display
import os
from threading import Lock

slide_lock = Lock()

image_filename = "ffcd99c47e57ad2934dc6bbf5edf6675.tiff"
image_path = f"/content/drive/MyDrive/Doutorado/kaggle/dataset/train_images/{image_filename}"
thumbnail_dir = "/content/drive/MyDrive/Doutorado/kaggle/dataset/thumbnails"

try:
    # Open the slide
    with slide_lock:
        slide = openslide.OpenSlide(f"{image_path}/{image_filename}")
        full_size = slide.level_dimensions[1]
        image = slide.read_region((0, 0), 1, full_size)
        slide.close()

    # Define thumbnail size
    thumbnail_size = (200, 200) # Max width, max height
    thumbnail = image.get_thumbnail(thumbnail_size)

    print(f"Thumbnail for {image_filename} created:")
    display(thumbnail)

    # Save the thumbnail
    os.makedirs(thumbnail_dir, exist_ok=True)
    thumbnail_output_path = f"{thumbnail_dir}/{image_filename.replace('.tiff', '_thumbnail.png')}"
    thumbnail.save(thumbnail_output_path)
    print(f"Thumbnail saved to {thumbnail_output_path}")

    slide.close()

except openslide.OpenSlideUnsupportedFormatError as e:
    print(f"Error: {e}. The file '{image_filename}' might be corrupted or in an unsupported format.")
    print("Please ensure the original .tiff file is valid and accessible.")
except FileNotFoundError:
    print(f"Error: The file '{image_path}' was not found.")
    print("Please ensure the image file exists at the specified path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Error: Unsupported or missing image file. The file 'ffcd99c47e57ad2934dc6bbf5edf6675.tiff' might be corrupted or in an unsupported format.
Please ensure the original .tiff file is valid and accessible.
